# Stage 2: Intermediate RAG — Hybrid Search & Reranking

Legal documents contain highly specific terminology (e.g. "liquidation preference", "indemnification cap"). Naive vector search can sometimes fail to match these exact keywords if they are represented differently in vector space. 

To solve this, Stage 2 introduces **Hybrid Search** (combining BM25 lexical match + Cosine semantic vectors) and **Reranking** (using Cohere client models).

### Learning Objectives
1. **Lexical Search**: Index chunks in a BM25 index.
2. **Vector Search**: Search via embeddings.
3. **Reciprocal Rank Fusion (RRF)**: Blend lexical and vector score matches.
4. **Rerank**: Use a cross-encoder model to re-evaluate the top matches against the prompt.

In [ ]:
# 1. Load Environment & Initialise Modules
from dotenv import load_dotenv
from src.parser import LegalDocumentParser
from src.embedder import LegalTextEmbedder
from src.vector_db import LegalVectorStore
from src.search import LegalHybridSearch

load_dotenv()
print("Stage 2 modules loaded successfully.")

In [ ]:
# 2. Parse & Chunk multiple contracts
parser = LegalDocumentParser()
nda_pages = parser.parse_pdf("data/raw/Mutual NDA.pdf")
emp_pages = parser.parse_pdf("data/raw/Employment Agreement.pdf")

all_chunks = parser.chunk_document(nda_pages) + parser.chunk_document(emp_pages)
print(f"Total parsed chunks across documents: {len(all_chunks)}")

In [ ]:
# 3. Setup Vector Store & Embeddings
embedder = LegalTextEmbedder(provider="openai")
vector_store = LegalVectorStore(db_type="faiss")

texts = [c["text"] for c in all_chunks]
embeddings = embedder.get_embeddings(texts)
vector_store.add_documents(all_chunks, embeddings)

In [ ]:
# 4. Fit BM25 & Initialize Hybrid Search
hybrid_search = LegalHybridSearch(vector_store, embedder)
hybrid_search.fit_bm25(all_chunks)

In [ ]:
# 5. Execute Hybrid Search (Lexical + Semantic blending)
query = "NDA exclusions and definitions of trade secrets"
hybrid_results = hybrid_search.hybrid_search(query, n_results=6)
print(f"Retrieved {len(hybrid_results)} hybrid results.")

In [ ]:
# 6. Rerank top results with Cohere Rerank
reranked_results = hybrid_search.rerank_results(query, hybrid_results, top_n=3)
print("Top 3 Reranked Clauses:\n")
for r in reranked_results:
    print(f"Document: {r['metadata']['document_name']} | Rerank Relevance Score: {r.get('rerank_score', 'N/A')}")
    print(r['text'][:400] + "...")
    print("-" * 80)